**<h1 align="center">ESCO Feature Construction</h1>**

This notebook constructs ontology-based features from the ESCO v1.1 dataset to support skill-aware job representation and analysis. Using essential skill relations only, it builds structured mappings between occupations and skills and computes corpus-level statistics used in later experiments. Specifically, the notebook:
- Loads ESCO occupations, skills, and occupation–skill relations
- Filters relations to essential skills
- Builds a mapping from each occupation to its associated essential skills
- Computes skill document frequency across occupations
- Derives inverse document frequency (IDF) weights for skills
- Computes an occupation-level skill specificity score (average skill IDF)
- Performs coverage and sanity checks on occupations and skills
- Saves all derived artifacts as reusable JSON files for downstream experiments


The outputs of this notebook provide ontology-derived signals that can be integrated into SBERT-based models or used for qualitative analysis, enabling controlled experiments on whether structured ESCO knowledge improves career transition prediction.

<a class="anchor" id="chapter1"></a>

# 1. Imports & Setup

</a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import json
import re
import sys
import math
import pandas as pd
from collections import defaultdict, Counter

In [ ]:
BASE = "/content/drive/MyDrive/NOVA IMS/Year 2/Thesis"
if BASE not in sys.path:
    sys.path.insert(0, BASE)

from thesis_utility import save_json

In [ ]:
DATA_DIR = f"{BASE}/Data"
OUT_DIR = f"{BASE}/Ontology Artifacts"

<a class="anchor" id="chapter2"></a>

# 2. Load Data

</a>

In [ ]:
dictionary = pd.read_csv(f"{DATA_DIR}/dictionary_en.csv")

In [ ]:
skills = pd.read_csv(f"{DATA_DIR}/skills_en.csv")
occ_skill = pd.read_csv(f"{DATA_DIR}/occupationSkillRelations_en.csv")
occ = pd.read_csv(f"{DATA_DIR}/occupations_en.csv")
hierarchy = pd.read_csv(f"{DATA_DIR}/skillsHierarchy_en.csv")

<a class="anchor" id="chapter3"></a>

# 3. Build lookup tables

</a>

In [ ]:
# Check dictionary columns

In [ ]:
print("skills_en.csv:", skills.columns)
print("occupationSkillRelations_en.csv:", occ_skill.columns)
print("occupations_en.csv:", occ.columns)

skills_en.csv: Index(['conceptType', 'conceptUri', 'skillType', 'reuseLevel',
       'preferredLabel', 'altLabels', 'hiddenLabels', 'status', 'modifiedDate',
       'scopeNote', 'definition', 'inScheme', 'description'],
      dtype='object')
occupationSkillRelations_en.csv: Index(['occupationUri', 'occupationLabel', 'relationType', 'skillType',
       'skillUri', 'skillLabel'],
      dtype='object')
occupations_en.csv: Index(['conceptType', 'conceptUri', 'iscoGroup', 'preferredLabel', 'altLabels',
       'hiddenLabels', 'status', 'modifiedDate', 'regulatedProfessionNote',
       'scopeNote', 'definition', 'inScheme', 'description', 'code',
       'naceCode'],
      dtype='object')


In [ ]:
essential_skills = occ_skill[occ_skill['relationType'] == 'essential']

In [ ]:
occ_to_label = occ.set_index('conceptUri')['preferredLabel'].to_dict()
occ_to_isco = occ.set_index('conceptUri')['iscoGroup'].to_dict()
occ_to_desc = occ.set_index('conceptUri')['description'].to_dict()

In [ ]:
skill_to_label = skills.set_index('conceptUri')['preferredLabel'].to_dict()
skill_to_type = skills.set_index('conceptUri')['skillType'].to_dict()

<a class="anchor" id="chapter4"></a>

# 4. Build skill hierarchy lookup (skill → parents)

</a>

The file provides paths Level 0 → Level 1 → Level 2 → Level 3 <br>
Convert these paths into direct parent links:
- Level1 parent = Level0
- Level2 parent = Level1
- Level3 parent = Level2

In [ ]:
hierarchy.columns

Index(['Level 0 URI', 'Level 0 preferred term', 'Level 1 URI',
       'Level 1 preferred term', 'Level 2 URI', 'Level 2 preferred term',
       'Level 3 URI', 'Level 3 preferred term', 'Description', 'Scope note',
       'Level 0 code', 'Level 1 code', 'Level 2 code', 'Level 3 code'],
      dtype='object')

In [ ]:
def build_skill_to_parents_from_levels(hierarchy_df: pd.DataFrame) -> dict:
    parent_map = defaultdict(set)

    # Ensure expected columns exist
    required = ["Level 0 URI", "Level 1 URI", "Level 2 URI", "Level 3 URI"]
    missing = [c for c in required if c not in hierarchy_df.columns]
    if missing:
        raise ValueError(f"skillsHierarchy_en.csv is missing columns: {missing}")

    for _, row in hierarchy_df.iterrows():
        l0 = row["Level 0 URI"]
        l1 = row["Level 1 URI"]
        l2 = row["Level 2 URI"]
        l3 = row["Level 3 URI"]

        # Add direct parent edges when both child and parent are present strings
        if isinstance(l0, str) and isinstance(l1, str) and l0.strip() and l1.strip():
            parent_map[l1.strip()].add(l0.strip())

        if isinstance(l1, str) and isinstance(l2, str) and l1.strip() and l2.strip():
            parent_map[l2.strip()].add(l1.strip())

        if isinstance(l2, str) and isinstance(l3, str) and l2.strip() and l3.strip():
            parent_map[l3.strip()].add(l2.strip())

    # Convert to JSON-serializable dict[str, list[str]]
    skill_to_parents = {child: sorted(list(parents)) for child, parents in parent_map.items()}
    return skill_to_parents

In [ ]:
skill_to_parents = build_skill_to_parents_from_levels(hierarchy)

In [ ]:
print("Skills with >=1 parent:", len(skill_to_parents))
# quick sanity check: show 5 examples
for i, (k, v) in enumerate(skill_to_parents.items()):
    if i == 5:
        break
    print("child:", k, "parents:", v)

Skills with >=1 parent: 636
child: http://data.europa.eu/esco/skill/43f425aa-f45d-4bb4-a200-6f82fa211b66 parents: ['http://data.europa.eu/esco/skill/e35a5936-091d-4e87-bafe-f264e55bd656']
child: http://data.europa.eu/esco/skill/e434e71a-f068-44ed-8059-d1af9eb592d7 parents: ['http://data.europa.eu/esco/skill/e35a5936-091d-4e87-bafe-f264e55bd656']
child: http://data.europa.eu/esco/skill/03e0b95b-67d1-457a-b3f7-06c407cf6bec parents: ['http://data.europa.eu/esco/skill/335228d2-297d-4e0e-a6ee-bc6a8dc110d9']
child: http://data.europa.eu/esco/skill/15dfca7a-5dde-4199-bad3-c00600387258 parents: ['http://data.europa.eu/esco/skill/03e0b95b-67d1-457a-b3f7-06c407cf6bec']
child: http://data.europa.eu/esco/skill/61d1dab2-6007-4b7c-9380-cd88207fa30f parents: ['http://data.europa.eu/esco/skill/15dfca7a-5dde-4199-bad3-c00600387258']


<a class="anchor" id="chapter4"></a>

# 5. Parse alternative labels

</a>

In [ ]:
def split_altlabels(x):
    if not isinstance(x, str):
        return []
    return [
        s.strip()
        for s in re.split(r'[;\n]', x)
        if s.strip()
    ]

In [ ]:
skill_to_altlabels = skills.set_index('conceptUri')['altLabels'].apply(split_altlabels).to_dict()

<a class="anchor" id="chapter5"></a>

# 6. Build occupation → essential skills map

</a>

In [ ]:
occ_to_skills = defaultdict(set)

for _, row in essential_skills.iterrows():
    occ_to_skills[row['occupationUri']].add(row['skillUri'])

In [ ]:
occ_skill_count = {occ: len(skills) for occ, skills in occ_to_skills.items()}

In [ ]:
# 1) One occupation
some_occ = next(iter(occ_to_skills))
print(occ_to_label[some_occ])
print(len(occ_to_skills[some_occ]))

# 2) One skill
some_skill = next(iter(next(iter(occ_to_skills.values()))))
print(skill_to_label[some_skill])
print(skill_to_altlabels[some_skill][:3])

technical director
9
adapt to artists' creative demands
["adapt to artists' creative vision", 'meet demands by creative artists', 'adapt to demands by creative artists']


<a class="anchor" id="chapter6"></a>

# 7. Compute IDF for skills (global specificity) & Occupation-level specificity (avg skill IDF)

</a>

In [ ]:
# occ_to_skills: dict[occ_uri] -> set[skill_uri]
N_occ = len(occ_to_skills)

# count in how many occupations each skill appears
df_counter = Counter()
for skills_set in occ_to_skills.values():
    df_counter.update(skills_set)

skill_df = dict(df_counter)

skill_idf = {
    s: math.log((N_occ + 1) / (df + 1)) + 1.0
    for s, df in skill_df.items()
}

# optional: occupation-level specificity
occ_avg_skill_idf = {
    occ: (sum(skill_idf[s] for s in skills_set) / max(1, len(skills_set)))
    for occ, skills_set in occ_to_skills.items()
}

<a class="anchor" id="chapter7"></a>

# 8. Coverage + diagnostics

</a>

In [ ]:
n_occ_total = occ['conceptUri'].nunique()
n_occ_with_skills = len(occ_to_skills)

n_skills_total = skills['conceptUri'].nunique()
n_skills_in_mapping = len(skill_df)

print("Occupations total:", n_occ_total)
print("Occupations with >=1 essential skill:", n_occ_with_skills,
      f"({n_occ_with_skills/n_occ_total:.2%})")

print("Skills total:", n_skills_total)
print("Skills appearing as essential at least once:", n_skills_in_mapping,
      f"({n_skills_in_mapping/n_skills_total:.2%})")

Occupations total: 3039
Occupations with >=1 essential skill: 3039 (100.00%)
Skills total: 13939
Skills appearing as essential at least once: 11378 (81.63%)


In [ ]:
occ_skill_counts = [len(s) for s in occ_to_skills.values()]
print("Essential skills per occupation:")
print("min / median / mean / max:",
      min(occ_skill_counts),
      sorted(occ_skill_counts)[len(occ_skill_counts)//2],
      sum(occ_skill_counts)/len(occ_skill_counts),
      max(occ_skill_counts))

Essential skills per occupation:
min / median / mean / max: 3 19 22.24415926291543 99


In [ ]:
# occupations referenced in relations but missing from occupations file
occ_set = set(occ['conceptUri'])
mapped_occ_set = set(occ_to_skills.keys())
missing_occ = mapped_occ_set - occ_set
print("Occupations in relations but not in occupations_en:", len(missing_occ))

# skills referenced in relations but missing from skills file
skill_set = set(skills['conceptUri'])
mapped_skill_set = set(skill_df.keys())
missing_skill = mapped_skill_set - skill_set
print("Skills in relations but not in skills_en:", len(missing_skill))

Occupations in relations but not in occupations_en: 0
Skills in relations but not in skills_en: 0


In [ ]:
n_skill_nodes_in_hierarchy = len(set(hierarchy["Level 0 URI"].dropna().astype(str))
                                 | set(hierarchy["Level 1 URI"].dropna().astype(str))
                                 | set(hierarchy["Level 2 URI"].dropna().astype(str))
                                 | set(hierarchy["Level 3 URI"].dropna().astype(str)))

print("Unique skill URIs appearing anywhere in hierarchy:", n_skill_nodes_in_hierarchy)
print("Skills that have at least one parent (child->parent edges):", len(skill_to_parents))

Unique skill URIs appearing anywhere in hierarchy: 640
Skills that have at least one parent (child->parent edges): 636


<a class="anchor" id="chapter9"></a>

# 9. Save Artifacts

</a>

In [ ]:
occ_to_skills_json = {
    occ: sorted(list(skills))
    for occ, skills in occ_to_skills.items()
}

In [ ]:
save_json(occ_to_skills_json, "occ_to_skills.json", out_dir=OUT_DIR)
save_json(skill_df, "skill_df.json", out_dir=OUT_DIR)
save_json(skill_idf, "skill_idf.json", out_dir=OUT_DIR)
save_json(occ_avg_skill_idf, "occ_avg_skill_idf.json", out_dir=OUT_DIR)
save_json(skill_to_parents, "skill_to_parents.json", out_dir=OUT_DIR)